In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [44]:
class LineRatePredictor(nn.Module):
    def __init__(self, num_features):
        super(LineRatePredictor, self).__init__()

        self.conv1 = nn.Conv1d(num_features, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=32,
            batch_first=True
        )

        self.fc1 = nn.Linear(32, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        # x shape: (batch, time, features)

        x = x.permute(0, 2, 1)       # → (batch, features, time)

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = x.permute(0, 2, 1)       # → (batch, time, channels)

        x, _ = self.lstm(x)

        x = x[:, -1, :]              # last timestep

        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))

        return x

In [45]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

In [ ]:
class SensorDataset(Dataset):

    def __init__(self, df, shift):
        self.reading = df
        self.shift = shift

    def __len__(self):
        return len(self.reading) - self.shift

    def __getitem__(self, idx):

        sensor_reading = torch.from_numpy(
            self.reading.iloc[idx, 1:4].to_numpy
        )

        label = torch.tensor(
            self.reading.iloc[idx + self.shift, -1],
            dtype=torch.float32
        )

        return sensor_reading, label
        
        

In [56]:
df = pd.read_csv("./sensor_dataset.csv")

dummy_set = SensorDataset(df, 5)
dummy_loader = DataLoader(dummy_set, batch_size=32, shuffle=True)

In [57]:
model = LineRatePredictor(3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [58]:
def train_model(model, train_loader, val_loader, epochs=50):

    for epoch in range(epochs):

        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            preds = model(X_batch).squeeze()

            loss = criterion(preds, y_batch)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                preds = model(X_batch).squeeze()

                loss = criterion(preds, y_batch)

                val_loss += loss.item()

                predicted = (preds > 0.5).float()

                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss /= len(val_loader)
        accuracy = correct / total

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {accuracy:.4f}"
        )

In [59]:
train_model(model=model, train_loader=dummy_loader, val_loader=dummy_loader)

TypeError: torch._VariableFunctionsClass.from_numpy() takes no keyword arguments

In [ ]:
!pwd

/content
